In [ ]:
# ============================================================
#  Colab: Traditional ML GPU/CPU pipeline + runner
# ============================================================

from __future__ import annotations

import argparse
import json
import os
import sys
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import itertools  # add this with your other imports

import numpy as np
import time

# sklearn + plotting
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.neighbors import KNeighborsClassifier as skKNN
from sklearn.svm import SVC as skSVC
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split


import joblib
import matplotlib
matplotlib.use("Agg")  # no interactive display needed
import matplotlib.pyplot as plt

# ------------------------------ Utilities ----------------------------------

def _try_import_cuml():
    """Best-effort import of cuML + CuPy stack; return None on failure."""
    try:
        import cupy as cp  # type: ignore
        from cuml.neighbors import KNeighborsClassifier as cuKNN  # type: ignore
        from cuml.svm import SVC as cuSVC  # type: ignore
        from cuml.ensemble import RandomForestClassifier as cuRF  # type: ignore
        return {"cp": cp, "cuKNN": cuKNN, "cuSVC": cuSVC, "cuRF": cuRF}
    except Exception:
        return None


def has_gpu() -> bool:
    """Best-effort GPU check without raising on CPU-only hosts."""
    try:
        import cupy as cp  # type: ignore
        _ = cp.cuda.runtime.getDeviceCount()  # raises if no CUDA
        return _ > 0
    except Exception:
        return False
def plot_confmat(cm: np.ndarray, class_names: Optional[List[str]], out_path: Path) -> None:
    """
    Confusion-matrix plot styled like the RNN script.
    Saves a PNG to `out_path`.
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title("Confusion Matrix")
    fig.colorbar(im, ax=ax)

    if class_names is None:
        class_names = [str(i) for i in range(cm.shape[0])]

    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    thresh = cm.max() / 2.0 if cm.max() > 0 else 1.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(
            j,
            i,
            format(cm[i, j], "d"),
            horizontalalignment="center",
            verticalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

    ax.set_ylabel("True")
    ax.set_xlabel("Pred")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def to_numpy(x: Any) -> np.ndarray:
    """Convert CuPy arrays to NumPy when needed."""
    try:
        import cupy as cp  # type: ignore
        if isinstance(x, cp.ndarray):
            return cp.asnumpy(x)
    except Exception:
        pass
    return np.asarray(x)


def maybe_to_cupy(arr: np.ndarray, use_gpu: bool) -> Any:
    """Move to GPU if requested and available."""
    if use_gpu:
        try:
            import cupy as cp  # type: ignore
            return cp.asarray(arr)
        except Exception:
            pass
    return arr
def plot_accuracy_summary(results: Dict[str, Dict[str, Any]], out_path: Path) -> None:
    """
    Bar chart of CV vs Test accuracy per model.
    Saves to out_path.
    """
    if not results:
        return

    models = sorted(results.keys())
    cv_acc = [results[m].get("cv_accuracy", float("nan")) for m in models]
    test_acc = [results[m].get("test_accuracy", float("nan")) for m in models]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width / 2, cv_acc, width, label="CV accuracy")
    ax.bar(x + width / 2, test_acc, width, label="Test accuracy")

    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in models])
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.0, 1.0)
    ax.set_title("Traditional ML – Accuracy Summary")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def plot_time_summary(results: Dict[str, Dict[str, Any]], out_path: Path) -> None:
    """
    Bar chart of train/test time per model (seconds), if timing info is present.
    Saves to out_path.
    """
    if not results:
        return

    # Only plot if at least one model has timing data
    if not any("train_time_sec" in r for r in results.values()):
        return

    models = sorted(results.keys())
    train_t = [results[m].get("train_time_sec", np.nan) for m in models]
    test_t = [results[m].get("test_time_sec", np.nan) for m in models]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(x - width / 2, train_t, width, label="Train time (s)")
    ax.bar(x + width / 2, test_t, width, label="Test time (s)")

    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in models])
    ax.set_ylabel("Seconds")
    ax.set_title("Traditional ML – Timing Summary")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


@dataclass
class DataBundle:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: Optional[np.ndarray] = None
    y_val: Optional[np.ndarray] = None
    X_test: Optional[np.ndarray] = None
    y_test: Optional[np.ndarray] = None
    snr_train: Optional[np.ndarray] = None
    snr_val: Optional[np.ndarray] = None
    snr_test: Optional[np.ndarray] = None


# ------------------------------ Data Loading (.npz) -------------------------

def load_data_npz(path: Path) -> DataBundle:
    """
    Load data from an .npz with keys:
      - X_train, y_train (required)
      - X_val, y_val, X_test, y_test (optional)
      - snr_train, snr_val, snr_test (optional, for SNR curves)
    """
    data = np.load(path, allow_pickle=True)  # allow object arrays (for saved None)

    def _maybe_none(name: str):
        if name not in data:
            return None
        arr = data[name]
        # If we saved None, this comes back as a 0-d object array with value None
        if isinstance(arr, np.ndarray) and arr.dtype == object and arr.shape == ():
            val = arr[()]
            return None if val is None else val
        return arr

    X_train = _maybe_none("X_train")
    y_train = _maybe_none("y_train")
    X_val   = _maybe_none("X_val")
    y_val   = _maybe_none("y_val")
    X_test  = _maybe_none("X_test")
    y_test  = _maybe_none("y_test")

    snr_train = _maybe_none("snr_train")
    snr_val   = _maybe_none("snr_val")
    snr_test  = _maybe_none("snr_test")

    return DataBundle(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        snr_train=snr_train,
        snr_val=snr_val,
        snr_test=snr_test,
    )

# ------------------------------ Models --------------------------------------

def build_model(name: str, use_gpu: bool) -> Tuple[Any, Dict[str, List[Any]]]:
    """
    Return (estimator, param_distributions) for RandomizedSearchCV or manual search (cuML).
    For cuML, some params names mirror sklearn where applicable.
    """
    name = name.lower()
    gpu_stack = _try_import_cuml() if use_gpu else None

    # ---------------- KNN ----------------
    if name == "knn":
        if gpu_stack:
            # cuML only supports uniform weights
            model = gpu_stack["cuKNN"]()
            params = {
                "n_neighbors": [3, 5, 7, 9],
            }
        else:
            model = skKNN()
            params = {
                "n_neighbors": [3, 5, 7, 9],
                "weights": ["uniform", "distance"],
            }
        return model, params

    # ---------------- SVM / SVC ----------------
    if name in ("svm", "svc"):
        if gpu_stack:
            # cuML SVC: use numeric gamma values (no "scale"/"auto")
            model = gpu_stack["cuSVC"](kernel="rbf")
            params = {
                "C": [0.5, 1, 2, 4, 8],
                "gamma": [0.1, 0.01, 0.001],
            }
        else:
            # sklearn SVC: "scale"/"auto" are fine
            model = skSVC(kernel="rbf", probability=False)
            params = {
                "C": [0.5, 1, 2, 4, 8],
                "gamma": ["scale", "auto"],
            }
        return model, params

    # ---------------- Random Forest ----------------
    if name in ("rf", "randomforest", "random_forest"):
        if gpu_stack:
            # cuML RF: avoid None in params (cuML compares against ints internally)
            model = gpu_stack["cuRF"](random_state=42, n_streams=4)
            params = {
                "n_estimators": [200, 400, 800],
                # cuML expects integers for max_depth; no None
                "max_depth": [12, 16, 20, 24],
                # keep only explicit values; avoid None here too
                "max_features": [0.5, "sqrt", "log2"],
            }
        else:
            # sklearn RF: "auto" and None are fine
            model = skRF(random_state=42, n_jobs=-1)
            params = {
                "n_estimators": [200, 400, 800],
                "max_depth": [None, 12, 16, 20, 24],
                "max_features": ["auto", "sqrt", 0.5],
            }
        return model, params


    # ---------------- Unknown ----------------
    raise ValueError(f"Unknown model: {name}")



# ------------------------------ Training ------------------------------------

def search_and_fit(
    name: str,
    X: np.ndarray,
    y: np.ndarray,
    use_gpu: bool,
    cv_splits: int,
    n_iter: int,
    random_state: int = 42,
) -> Tuple[Any, Dict[str, Any], float, float]:
    """
    Hyperparameter search + CV.
    - On CPU: RandomizedSearchCV with scaler for SVM/KNN.
    - On GPU: simple manual randomized search to avoid sklearn<->cuML compatibility pitfalls.
    Returns (best_estimator, best_params, best_cv_score, train_time_sec).
    """
    start_time = time.perf_counter()

    rng = np.random.default_rng(random_state)
    est, param_space = build_model(name, use_gpu=use_gpu)

    # Scaling for distance-based models/SVM
    needs_scaler = name.lower() in {"knn", "svm", "svc"}

    if not use_gpu:
        pipe_steps = []
        if needs_scaler:
            pipe_steps.append(("scaler", StandardScaler()))
        pipe_steps.append(("model", est))
        pipe = SkPipeline(pipe_steps)
        # Map param grid into pipeline namespace
        param_distributions = {f"model__{k}": v for k, v in param_space.items()}

        cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=param_distributions,
            n_iter=min(n_iter, sum(len(v) for v in param_distributions.values())),
            scoring="accuracy",
            cv=cv,
            n_jobs=-1,
            random_state=random_state,
            verbose=1,
        )
        search.fit(X, y)
        best = search.best_estimator_
        train_time = time.perf_counter() - start_time
        return best, search.best_params_, float(search.best_score_), train_time

    # GPU path: manual random search to avoid sklearn CV overhead converting arrays back/forth
    gpu = _try_import_cuml()
    assert gpu is not None, "GPU stack not available despite use_gpu=True"
    cp = gpu["cp"]

    # Apply scaling manually if needed (on CPU to avoid mismatch; data size usually manageable)
    X_scaled = X
    scaler = None
    if needs_scaler:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

    # Convert to GPU
    X_gpu = cp.asarray(X_scaled)
    y_gpu = cp.asarray(y)

    # Generate param samples
    grid_kv = list(param_space.items())

    def sample_params() -> Dict[str, Any]:
        params = {}
        for k, values in grid_kv:
            idx = rng.integers(len(values))   # <- keep original Python types
            params[k] = values[idx]
        return params

    best_score, best_params, best_model = -np.inf, None, None
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    for _ in range(n_iter):
        params = sample_params()
        # Rebuild estimator each time to avoid state carryover
        est, _ = build_model(name, use_gpu=True)
        est.set_params(**params)
        fold_scores: List[float] = []
        for train_idx, val_idx in cv.split(X, y):
            Xtr, Xva = X_gpu[train_idx], X_gpu[val_idx]
            ytr, yva = y_gpu[train_idx], y_gpu[val_idx]
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)
            pred_np = to_numpy(pred)
            yva_np = to_numpy(yva)
            fold_scores.append(accuracy_score(yva_np, pred_np))
        mean_score = float(np.mean(fold_scores))
        if mean_score > best_score:
            best_score, best_params = mean_score, params
            best_model = est

    if needs_scaler:
        best = ("scaler+gpu_model", scaler, best_model)
    else:
        best = ("gpu_model", best_model)

    train_time = time.perf_counter() - start_time
    return best, best_params, float(best_score), train_time

def predict_with(est: Any, X: np.ndarray, use_gpu: bool) -> np.ndarray:
    """Predict with the possibly composite estimator returned above."""
    if not use_gpu:
        return est.predict(X)

    tag = est[0]
    if tag == "gpu_model":
        model = est[1]
        gpu = _try_import_cuml()
        cp = gpu["cp"]
        return to_numpy(model.predict(cp.asarray(X)))
    elif tag == "scaler+gpu_model":
        _, scaler, model = est
        Xs = scaler.transform(X)
        gpu = _try_import_cuml()
        cp = gpu["cp"]
        return to_numpy(model.predict(cp.asarray(Xs)))
    raise ValueError("Unknown estimator wrapper")


# ------------------------------ Evaluation ----------------------------------

def evaluate_and_log(
    est: Any,
    X_test: np.ndarray,
    y_test: np.ndarray,
    out_dir: Path,
    label_names: Optional[List[str]] = None,
    use_gpu: bool = False,
    prefix: str = "",
) -> Dict[str, Any]:
    """Compute metrics, save JSON + confusion matrix image."""
    ensure_dir(out_dir)
    y_pred = predict_with(est, X_test, use_gpu=use_gpu)

    acc = float(accuracy_score(y_test, y_pred))
    report = classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "accuracy": acc,
        "report": report,
        "confusion_matrix": cm.tolist(),
    }
    (out_dir / f"{prefix}metrics.json").write_text(json.dumps(metrics, indent=2))

    # Use shared plot_confmat style; save into this model's folder
    class_names = label_names if label_names is not None else [str(i) for i in range(cm.shape[0])]
    plot_confmat(
        cm=cm,
        class_names=class_names,
        out_path=out_dir / f"{prefix}confusion.png",
    )

    return {"accuracy": acc, "report": report}

def compute_snr_accuracy_curve(
    est: Any,
    X_test: np.ndarray,
    y_test: np.ndarray,
    snr_test: np.ndarray,
    use_gpu: bool,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute *test-set* accuracy vs SNR for a single model.
    Returns (snr_values, accuracies) where both are sorted by SNR.
    """
    snr_values = np.unique(snr_test)
    accs = []
    for snr_val in snr_values:
        idx = snr_test == snr_val
        if not np.any(idx):
            continue
        X_snr = X_test[idx]
        y_true = y_test[idx]
        y_pred = predict_with(est, X_snr, use_gpu=use_gpu)
        accs.append(accuracy_score(y_true, y_pred))
    return snr_values, np.array(accs, dtype=float)


def plot_snr_curves(
    snr_curves: Dict[str, Dict[str, List[float]]],
    out_path: Path,
) -> None:
    """
    Plot *test accuracy* vs SNR curves for multiple models.
    snr_curves[name] = {"snr": [...], "accuracy": [...]}
    """
    if not snr_curves:
        return

    fig, ax = plt.subplots(figsize=(6, 4))
    for name, curve in snr_curves.items():
        snr_vals = np.array(curve["snr"])
        acc_vals = np.array(curve["accuracy"])
        ax.plot(snr_vals, acc_vals, marker="o", label=name.upper())

    ax.set_xlabel("SNR (dB)")
    ax.set_ylabel("Test Accuracy")
    ax.set_title("Test Accuracy vs SNR – Traditional ML Models")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


# ------------------------------ CLI-style main ------------------------------

def main(argv: Optional[List[str]] = None) -> int:
    p = argparse.ArgumentParser(description="CPU/GPU classical ML training pipeline")
    p.add_argument(
        "--data",
        type=Path,
        required=True,
        help=".npz with X_train,y_train,(X_val,y_val),(X_test,y_test)",
    )
    p.add_argument(
        "--models",
        type=str,
        default="knn,svm,rf",
        help="Comma-separated list: knn,svm,rf",
    )
    p.add_argument("--device", type=str, choices=["auto", "cpu", "gpu"], default="auto")
    p.add_argument("--cv-splits", type=int, default=5)
    p.add_argument(
        "--n-iter",
        type=int,
        default=20,
        help="Hyperparam samples per model",
    )
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--out", type=Path, default=Path("artifacts"))
    p.add_argument(
        "--labels",
        type=Path,
        help="Optional txt file with class labels, one per line",
    )
    args = p.parse_args(argv)

    use_gpu = {"auto": has_gpu(), "cpu": False, "gpu": True}[args.device]

    data = load_data_npz(args.data)
    ensure_dir(args.out)
    labels = None
    if args.labels and args.labels.exists():
        labels = [
            ln.strip()
            for ln in args.labels.read_text().splitlines()
            if ln.strip()
        ]

    # Merge train+val if provided; keep test separate
    X_train, y_train = data.X_train, data.y_train
    if data.X_val is not None and data.y_val is not None:
        X_train = np.concatenate([X_train, data.X_val], axis=0)
        y_train = np.concatenate([y_train, data.y_val], axis=0)

    X_test = data.X_test if data.X_test is not None else data.X_train
    y_test = data.y_test if data.y_test is not None else data.y_train

    selected = [m.strip().lower() for m in args.models.split(",") if m.strip()]
    results = {}
    model_objs: Dict[str, Any] = {}
    model_uses_gpu: Dict[str, bool] = {}

    for name in selected:
        # Use GPU for all models when available (KNN, SVM, RF via cuML)
        model_use_gpu = use_gpu
        model_uses_gpu[name] = model_use_gpu

        print(f"\n=== Training {name.upper()} on {'GPU' if model_use_gpu else 'CPU'} ===")
        est, best_params, cv_score, train_time = search_and_fit(
            name=name,
            X=X_train,
            y=y_train,
            use_gpu=model_use_gpu,
            cv_splits=args.cv_splits,
            n_iter=args.n_iter,
            random_state=args.seed,
        )
        print(f"Best CV accuracy for {name}: {cv_score:.4f} with params: {best_params}")
        print(f"Training time for {name}: {train_time:.2f} seconds")

        model_dir = args.out / name
        ensure_dir(model_dir)

        # Persist estimator
        joblib.dump(est, model_dir / "model.pkl")

        # Evaluate with timing
        eval_start = time.perf_counter()
        metrics = evaluate_and_log(
            est=est,
            X_test=X_test,
            y_test=y_test,
            out_dir=model_dir,
            label_names=labels,
            use_gpu=model_use_gpu,
            prefix="",
        )
        test_time = time.perf_counter() - eval_start
        print(f"Test accuracy for {name}: {metrics['accuracy']:.4f}")
        print(f"Test eval time for {name}: {test_time:.2f} seconds")

        results[name] = {
            "cv_accuracy": cv_score,
            "test_accuracy": metrics["accuracy"],
            "best_params": best_params,
            "train_time_sec": train_time,
            "test_time_sec": test_time,
        }
        model_objs[name] = est
    # Compute test accuracy vs SNR curves if we have snr_test in the bundle
    snr_curves: Dict[str, Dict[str, List[float]]] = {}
    if getattr(data, "snr_test", None) is not None:
        snr_test = data.snr_test
        for name, est in model_objs.items():
            model_use_gpu = model_uses_gpu[name]
            snr_vals, acc_vals = compute_snr_accuracy_curve(
                est=est,
                X_test=X_test,
                y_test=y_test,
                snr_test=snr_test,
                use_gpu=model_use_gpu,
            )
            snr_curves[name] = {
                "snr": [int(s) for s in snr_vals],
                "accuracy": [float(a) for a in acc_vals],
            }
            results[name]["snr_curve"] = snr_curves[name]

        snr_plot_path = args.out / "summary_snr_accuracy.png"
        plot_snr_curves(snr_curves, snr_plot_path)
        print(f"Saved SNR accuracy plot to: {snr_plot_path}")


    # Save summary
    # Save summary JSON
    summary_json_path = args.out / "summary.json"
    summary_json_path.write_text(json.dumps(results, indent=2))

    # Plot summary figures in the same root output folder
    plot_accuracy_summary(results, args.out / "summary_accuracy.png")
    plot_time_summary(results, args.out / "summary_time.png")

    print("\n=== Summary ===")
    print(json.dumps(results, indent=2))
    print(f"\nSaved summary JSON to: {summary_json_path}")
    print(f"Saved accuracy plot to: {args.out / 'summary_accuracy.png'}")
    print(f"Saved timing plot to: {args.out / 'summary_time.png'}")

    return 0



# ============================================================
#  Colab-specific runner: load .pkl, flatten, save .npz, run
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DATA_PKL_PATH = "/content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl"
OUT_BASE = Path("/content/drive/MyDrive/amc_runs/traditional_models")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# Data + labels saved into the same folder
NPZ_PATH = OUT_BASE / "RadioML2016_10A_traditional_flat.npz"
LABELS_TXT_PATH = OUT_BASE / "RadioML2016_10A_labels.txt"

# Artifacts (models, metrics, confusion matrices) – you can just reuse OUT_BASE
OUT_ARTIFACTS = OUT_BASE        # or OUT_BASE / "artifacts" if you want a subfolder
OUT_ARTIFACTS.mkdir(parents=True, exist_ok=True)


print("Loading pickle:", DATA_PKL_PATH)
with open(DATA_PKL_PATH, "rb") as f:
    # Use latin1 to be compatible with Python 2–style pickles / non-ASCII bytes
    raw_data = pickle.load(f, encoding="latin1")



def extract_splits(raw_data):
    """
    Convert RadioML2016.10A dict into:
      X_train, y_train, X_val, y_val, X_test, y_test,
      snr_train, snr_val, snr_test, label_names
    """
    if not isinstance(raw_data, dict):
        raise TypeError("Expected dict for RadioML2016.10A; got: {}".format(type(raw_data)))

    mods = sorted({k[0] for k in raw_data.keys()})
    mod_to_idx = {m: i for i, m in enumerate(mods)}

    X_list, y_list, snr_list = [], [], []

    for (mod, snr), arr in raw_data.items():
        if arr is None:
            continue
        arr = np.asarray(arr)
        if arr.ndim != 3 or arr.shape[1:] != (2, 128):
            raise ValueError(f"Unexpected array shape for key {(mod, snr)}: {arr.shape}")

        n = arr.shape[0]
        X_list.append(arr)
        y_list.append(np.full(n, mod_to_idx[mod], dtype=np.int64))
        snr_list.append(np.full(n, snr, dtype=np.int64))

    X   = np.concatenate(X_list, axis=0)
    y   = np.concatenate(y_list, axis=0)
    snr = np.concatenate(snr_list, axis=0)

    print("Built full dataset from dict:")
    print("  X shape:", X.shape)
    print("  y shape:", y.shape)
    print("  unique mods:", mods)
    print("  SNR range:", snr.min(), "to", snr.max())

    # Train/val/test: 70 / 15 / 15, stratified on y
    X_temp, X_test, y_temp, y_test, snr_temp, snr_test = train_test_split(
        X,
        y,
        snr,
        test_size=0.15,
        stratify=y,
        random_state=42,
    )

    X_train, X_val, y_train, y_val, snr_train, snr_val = train_test_split(
        X_temp,
        y_temp,
        snr_temp,
        test_size=0.1764706,  # 0.15 / 0.85
        stratify=y_temp,
        random_state=43,
    )

    label_names = mods
    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        snr_train,
        snr_val,
        snr_test,
        label_names,
    )


X_train, y_train, X_val, y_val, X_test, y_test, snr_train, snr_val, snr_test, label_names = extract_splits(raw_data)

print("Shapes BEFORE flattening:")
for name, arr in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),
]:
    print(f"  {name}: {None if arr is None else arr.shape}")


def flatten_if_needed(X):
    if X is None:
        return None
    # e.g. (N, 2, 128) -> (N, 256)
    return X.reshape(X.shape[0], -1)


X_train_f = flatten_if_needed(X_train)
X_val_f   = flatten_if_needed(X_val)
X_test_f  = flatten_if_needed(X_test)

print("Shapes AFTER flattening:")
for name, arr in [
    ("X_train_f", X_train_f),
    ("X_val_f", X_val_f),
    ("X_test_f", X_test_f),
]:
    print(f"  {name}: {None if arr is None else arr.shape}")

print("Saving NPZ to:", NPZ_PATH)
np.savez_compressed(
    NPZ_PATH,
    X_train=X_train_f,
    y_train=y_train,
    X_val=X_val_f,
    y_val=y_val,
    X_test=X_test_f,
    y_test=y_test,
    snr_train=snr_train,
    snr_val=snr_val,
    snr_test=snr_test,
)


labels_txt_path = None
if label_names is not None:
    print("Saving labels to:", LABELS_TXT_PATH)
    with open(LABELS_TXT_PATH, "w") as f:
        for name in label_names:
            f.write(str(name) + "\n")
    labels_txt_path = LABELS_TXT_PATH

# ---------------- Run the pipeline -----------------------------------------

argv = [
    "--data", str(NPZ_PATH),
    "--models", "knn,svm,rf",  # e.g. "svm,rf" if you want to subset
    "--device", "auto",        # "auto", "cpu", or "gpu"
    "--cv-splits", "5",
    "--n-iter", "20",
    "--seed", "42",
    "--out", str(OUT_ARTIFACTS),
]

if labels_txt_path is not None:
    argv.extend(["--labels", str(labels_txt_path)])

print("\nRunning traditional ML pipeline with args:")
print("  ", " ".join(argv))

exit_code = main(argv)
print("Pipeline finished with exit code:", exit_code)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading pickle: /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl
Built full dataset from dict:
  X shape: (220000, 2, 128)
  y shape: (220000,)
  unique mods: ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']
  SNR range: -20 to 18
Shapes BEFORE flattening:
  X_train: (153999, 2, 128)
  X_val: (33001, 2, 128)
  X_test: (33000, 2, 128)
Shapes AFTER flattening:
  X_train_f: (153999, 256)
  X_val_f: (33001, 256)
  X_test_f: (33000, 256)
Saving NPZ to: /content/drive/MyDrive/amc_runs/traditional_models/RadioML2016_10A_traditional_flat.npz
Saving labels to: /content/drive/MyDrive/amc_runs/traditional_models/RadioML2016_10A_labels.txt

Running traditional ML pipeline with args:
   --data /content/drive/MyDrive/amc_runs/traditional_models/RadioML2016_10A_traditional_flat.npz --models knn,svm,rf --device au